# AModule 1 Part 2: Agents

In [1]:
import os
import json
from dotenv import dotenv_values

from rag_helper import RAGBase
from ingest import load_faq_data, build_index
from sqlitesearch import TextSearchIndex

from openai import OpenAI

In [2]:
secrets_dir = os.path.expanduser("~/Documents/.secrets/llm-zoomcamp/")

config = {
    **dotenv_values(secrets_dir + "/.env.openai"),
}

api_key = config.get("OPENAI_API_KEY")
openai_client = OpenAI(api_key=api_key)

In [3]:
documents = load_faq_data()
index = build_index(documents)

In [4]:
index.search('how do I run ollama?')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

## Without tools...

In [5]:
message_history = [
    {"role": "user", "content": "Can I still join the course?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history,
)

response.output_text

'Yes—**maybe**. It depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out. Please send me:\n- the **course name**\n- the **school/platform**\n- whether you mean **late enrollment**, **waitlist**, or **adding after the deadline**\n\nIf you’re asking how to phrase it, you could message the instructor or registrar:\n\n> Hi, I’d like to ask whether it’s still possible to join the course. Is enrollment still open, or is there a waitlist/late-add option?\n\n'

In [6]:
response.output

[ResponseOutputMessage(id='msg_0f193e70f792b312006a323ecf2ebc81a0a4afb02cb549c575', content=[ResponseOutputText(annotations=[], text='Yes—**maybe**. It depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out. Please send me:\n- the **course name**\n- the **school/platform**\n- whether you mean **late enrollment**, **waitlist**, or **adding after the deadline**\n\nIf you’re asking how to phrase it, you could message the instructor or registrar:\n\n> Hi, I’d like to ask whether it’s still possible to join the course. Is enrollment still open, or is there a waitlist/late-add option?\n\n', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

## With tools...

First we define the tool. We do this in a language agnostic way, i.e. we always use JSON.

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I still join the course late enrollment deadline register after course start"}', call_id='call_FroAyRFQOvQrJnay2JDntCzv', name='search', type='function_call', id='fc_004ee6e35dc6e377006a323ed175d881a38917335c7e4ab15e', namespace=None, status='completed')]

Instead of returning a `ResponseOutputMessage` we return a `ResponseFunctionToolCall`

### Executing the function and sending back the result

In [10]:
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"Can I still join the course late enrollment deadline register after course start"}', call_id='call_FroAyRFQOvQrJnay2JDntCzv', name='search', type='function_call', id='fc_004ee6e35dc6e377006a323ed175d881a38917335c7e4ab15e', namespace=None, status='completed')

In [11]:
call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [12]:
message_history.extend(response.output)

In [13]:
message_history.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
    })

In [14]:
message_history

[{'role': 'user', 'content': 'Can I still join the course?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I still join the course late enrollment deadline register after course start"}', call_id='call_FroAyRFQOvQrJnay2JDntCzv', name='search', type='function_call', id='fc_004ee6e35dc6e377006a323ed175d881a38917335c7e4ab15e', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_FroAyRFQOvQrJnay2JDntCzv',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answe

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course and start learning anytime.\n\nA couple of notes:\n- The videos and materials are available.\n- You can submit homework while the submission form is open.\n- If you want a certificate, you need to complete the required project during the active course period, while submissions are still being accepted.\n\nIf you want, I can also tell you how to start the course and what the weekly workflow looks like.'

In [16]:
response.usage

ResponseUsage(input_tokens=810, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=91, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=901)

In [17]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0010170000000000001

## The Agentic Loop

The agent loop will be formed of 3 main parts:
- Instructions
- Tools
- Memory

In [18]:
# The starting point - instructions

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "Can I still join the course?"

message_history = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]


In [19]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=message_history,
    tools=[search_tool]
)



In [20]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join the course late enrollment can I still join the course"}', call_id='call_uSt5X9rosuCckGaJu02oFNWx', name='search', type='function_call', id='fc_0267e3a0f93601d5006a323ed4c3fc819cadd67958f31952d0', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment add drop late join registration FAQ"}', call_id='call_ChFj5vzaogzpXZCjGOAWBP6K', name='search', type='function_call', id='fc_0267e3a0f93601d5006a323ed4c40c819ca666d35989e09be7', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I still join the course after start date FAQ"}', call_id='call_sBUgdW0INK1Ltt8CPznhn7g0', name='search', type='function_call', id='fc_0267e3a0f93601d5006a323ed4c414819cbd7a0c647d2ada31', namespace=None, status='completed')]

We can create a make call function to take the actions that we completed for a single function call:
- 

In [21]:
def make_call(call):
    """
    Function to make tool call and return result to llm.
    - call: List[ResponseOutputItem]
    Returns dict to be added to message history
    """
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)
    
    result_json = json.dumps(results, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [22]:
message_history.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function:', item.name, item.arguments)
        function_call_output = make_call(item)
        message_history.append(function_call_output)

    elif item.type == 'message':
        print(item.content[0].text)

function: search {"query":"join the course late enrollment can I still join the course"}
function: search {"query":"course enrollment add drop late join registration FAQ"}
function: search {"query":"can I still join the course after start date FAQ"}


In [23]:
message_history

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'Can I still join the course?'},
 ResponseFunctionToolCall(arguments='{"query":"join the course late enrollment can I still join the course"}', call_id='call_uSt5X9rosuCckGaJu02oFNWx', name='search', type='function_call', id='fc_0267e3a0f93601d5006a323ed4c3fc819cadd67958f31952d0', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment add drop late join registration FAQ"}', call_id='call_ChFj5vzaogzpXZCjGOAWBP6K', n

In [24]:
message_history = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:

    print(f"iteration #{it}...")

    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=message_history,
        tools=[search_tool]
    )

    message_history.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function:', item.name, item.arguments)
            function_call_output = make_call(item)
            message_history.append(function_call_output)
            has_function_calls = True

        elif item.type == 'message':
            print(item.content[0].text)

    it += 1
    if has_function_calls == False:
        break

iteration #1...
function: search {"query":"join the course enrollment late registration can I still join the course"}
function: search {"query":"course enrollment add drop late join registration deadline"}
function: search {"query":"can I still enroll in the course after start date FAQ"}
iteration #2...
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. You can also start learning and submitting homework as long as the course forms/deadlines are still available.

If you'd like, I can also help with:
- how to start the course,
- whether you can still get a certificate,
- or how the homework/capstone works.


In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform a search, analyse the results and then perform additional searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [26]:
message_history = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:

    print(f"iteration #{it}...")

    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=message_history,
        tools=[search_tool]
    )

    message_history.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function:', item.name, item.arguments)
            function_call_output = make_call(item)
            message_history.append(function_call_output)
            has_function_calls = True

        elif item.type == 'message':
            print(item.content[0].text)

    it += 1
    if has_function_calls == False:
        break

iteration #1...
function: search {"query":"join course enrollment still join course late registration FAQ"}
iteration #2...
function: search {"query":"course still join accepted without registering while form is open certificate project submission live cohort self-paced FAQ"}
iteration #3...
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. You can also start learning and submitting homework while the forms/deadlines are open, even if you didn’t register.

If you want, I can also help you with the course workflow or certificate requirements.


In [28]:
def agent_loop(instructions, question, model='gpt-5.4-mini'):

    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1
    has_function_calls = True
    
    while has_function_calls:

        has_function_calls = False

        print(f"iteration #{it}...")

        response = openai_client.responses.create(
            model=model,
            input=message_history,
            tools=[search_tool]
        )

        message_history.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function:', item.name, item.arguments)
                function_call_output = make_call(item)
                message_history.append(function_call_output)
                has_function_calls = True

            elif item.type == 'message':
                final_answer = item.content[0].text
                print(final_answer)
                
        it += 1

    return final_answer
    

In [29]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function: search {"query":"Olama run locally installation local Ollama"}
iteration #2...
function: search {"query":"Ollama local run install model serve localhost course FAQ"}
iteration #3...
I couldn’t find a course FAQ entry specifically about **Ollama** (or “Olama”) local setup.

If you mean **Ollama**, the usual local run flow is:

1. **Install Ollama** from https://ollama.com
2. **Start the service**  
   - On many systems it runs automatically after installation.
   - You can also launch it from the app/terminal depending on your OS.
3. **Pull a model**
   ```bash
   ollama pull llama3.2
   ```
4. **Run it locally**
   ```bash
   ollama run llama3.2
   ```
5. **Use the local API**  
   Ollama typically serves on:
   ```text
   http://localhost:11434
   ```

Example API call:
```bash
curl http://localhost:11434/api/generate -d '{
  "model": "llama3.2",
  "prompt": "Hello!"
}'
```

If you meant something else by “Olama,” tell me the exact tool/package name and I’ll 

'I couldn’t find a course FAQ entry specifically about **Ollama** (or “Olama”) local setup.\n\nIf you mean **Ollama**, the usual local run flow is:\n\n1. **Install Ollama** from https://ollama.com\n2. **Start the service**  \n   - On many systems it runs automatically after installation.\n   - You can also launch it from the app/terminal depending on your OS.\n3. **Pull a model**\n   ```bash\n   ollama pull llama3.2\n   ```\n4. **Run it locally**\n   ```bash\n   ollama run llama3.2\n   ```\n5. **Use the local API**  \n   Ollama typically serves on:\n   ```text\n   http://localhost:11434\n   ```\n\nExample API call:\n```bash\ncurl http://localhost:11434/api/generate -d \'{\n  "model": "llama3.2",\n  "prompt": "Hello!"\n}\'\n```\n\nIf you meant something else by “Olama,” tell me the exact tool/package name and I’ll help you with the correct setup. Would you like help with running it in Docker, on Windows/macOS/Linux, or connecting it to Python?'

In [30]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function: search {"query":"queen gambit chess definition opening"}
iteration #2...
function: search {"query":"queen's gambit chess opening accepted declined pawn sacrifice"}
iteration #3...
The **Queen’s Gambit** is a classic chess opening.

It starts with:
1. **d4 d5**
2. **c4**

White offers the **c-pawn** as a “gambit” to try to:
- gain control of the center,
- open lines for development,
- and put pressure on Black’s d-pawn.

A few common versions:
- **Queen’s Gambit Accepted**: Black takes the c-pawn
- **Queen’s Gambit Declined**: Black does not take it

If you want, I can also explain:
- why it’s called a gambit,
- the main ideas for White and Black,
- or show a simple example line.


'The **Queen’s Gambit** is a classic chess opening.\n\nIt starts with:\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** as a “gambit” to try to:\n- gain control of the center,\n- open lines for development,\n- and put pressure on Black’s d-pawn.\n\nA few common versions:\n- **Queen’s Gambit Accepted**: Black takes the c-pawn\n- **Queen’s Gambit Declined**: Black does not take it\n\nIf you want, I can also explain:\n- why it’s called a gambit,\n- the main ideas for White and Black,\n- or show a simple example line.'

In [31]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function: search {"query":"queen gambit Queen's Gambit chess opening"}
iteration #2...
function: search {"query":"queen gambit queen's gambit FAQ chess opening"}
iteration #3...
I couldn’t find anything in the course FAQ about “queen gambit,” so it looks off-topic for this course.

If you meant something course-related, please rephrase it and I’ll look it up. Are there other areas you want to explore?


'I couldn’t find anything in the course FAQ about “queen gambit,” so it looks off-topic for this course.\n\nIf you meant something course-related, please rephrase it and I’ll look it up. Are there other areas you want to explore?'

### Putting this into a function

In [32]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform a search, analyse the results and then perform additional searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "Can I still join the course?"

agent_loop(instructions, question)

iteration #1...
function: search {"query":"join the course still enroll late registration course FAQ"}
iteration #2...
function: search {"query":"certificate still join course project submissions while accepting submissions registration just start learning homework while form is open"}
iteration #3...
Yes — you can still join the course.

From the FAQ: if you just discovered the course, you can start learning and submit homework while the submission form is open. Registration isn’t required to begin. If you want a certificate, you’ll need to submit your project while submissions are still being accepted.

If you want, I can also explain how to start the course or what you need for the certificate.


'Yes — you can still join the course.\n\nFrom the FAQ: if you just discovered the course, you can start learning and submit homework while the submission form is open. Registration isn’t required to begin. If you want a certificate, you’ll need to submit your project while submissions are still being accepted.\n\nIf you want, I can also explain how to start the course or what you need for the certificate.'

In [33]:
# Cleaned agent loop without print statements

def clean_agent_loop(instructions, question, model='gpt-5.4-mini'):

    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1
    has_function_calls = True
    
    while has_function_calls:

        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=message_history,
            tools=[search_tool]
        )

        message_history.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                function_call_output = make_call(item)
                message_history.append(function_call_output)
                has_function_calls = True

            elif item.type == 'message':
                final_answer = item.content[0].text
                
        it += 1

    return final_answer

In [34]:
answer = clean_agent_loop(instructions, question)
print(answer)

Yes — you can still join the course.

From the course FAQ: if you just discovered it, you’re still welcome to start. You can begin learning and submitting homework while the submission form is open. If you want a certificate, make sure you submit your project before the course stops accepting submissions.

If you want, I can also help you with:
- how to start the course,
- whether you can still get a certificate,
- or the weekly workflow.


## Implementation with ToyAIKit framework

This framework is not for production, just for learning!

In [35]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [36]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

If we document the function tool properly with a docstring and type hints we only need to provide the tool function, not additionally the function description JSON object.

In [37]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [38]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [39]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

## The chat interface and runner

In [42]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [43]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini',client=openai_client)
)

In [45]:
result = runner.loop(
    "Do I need a GPU to complete the course", 
    callback=callback,
    )

-> Response received


-> Response received


-> Response received


-> Response received


In [46]:
result.cost

CostInfo(input_cost=Decimal('0.003885'), output_cost=Decimal('0.0009'), total_cost=Decimal('0.004785'))

In [48]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform a search, analyse the results and then perform additional searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Do I need a GPU to complete the course', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"GPU required complete course FAQ"}', call_id='call_clwBDZS2rYy21cd6UPETdRku', name='search', type='function_call', id='fc_070207bad15fad86006a3242850c0c81a182039c0f3191cb03', namespace=None, status='completed'),
 {'type': 'functio

We can continue the conversation by adding the previous mnessages as a parameter when calling the `runner.loop` again

In [49]:
second_results = runner.loop(
    "Do I need cloud resources for this course?", 
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


-> Response received


To run the loop interactively we can use `runner.run();`

In [50]:
runner.run();

-> Response received


-> Response received


-> Response received


Chat ended.
